# M7a tile classifier — verification harness

This notebook is **not** an exploratory analysis. It is the executable half of §12 of
`docs_hpc/plan_addendum_m7a_tile_classifier.md`: every claim §11 makes about the six Step 9 runs is
re-derived here from the artifacts on disk and asserted, so that a number in the plan cannot drift
away from the number in the file without this notebook failing.

Nothing here reads the plan's prose. The checks read `results.json`, `predictions.csv`,
`batch_composition.jsonl` and `data/splits/split_manifest.json`, and where a value can be
recomputed it is recomputed independently rather than accepted.

| Cell | Discharges | Question it answers |
|------|-----------|---------------------|
| Check 1 | §12 item 1 | Does every results file carry exactly the key set §7 documents? |
| Check 2 | §12 item 2 | Does every reported metric re-derive from `predictions.csv`? |
| Check 3 | §12 item 3 | Does the observed batch composition match the requested ratio? |
| Check 4 | §12 item 4 | Is every run scored against the *current* split manifest? |
| Check 5 | §12 item 5 | Do the label counts reproduce from geometry at the recorded threshold? |
| Check 6 | §12 item 6 | Do the confidence intervals resample **specimens**? |
| Check 7 | §8 Step 8 | Is the operating point the exact validation-F1 maximiser, and is test scored at it? |

A failure is an `AssertionError` with the two disagreeing values in the message. Run top to bottom;
the last cell prints the summary tables that §11 quotes.

In [ ]:
"""Setup: locate the repository, discover every evaluated run, load its artifacts."""

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The kernel starts in notebooks/, so the repo root is one level up. Resolved by looking
# for a marker rather than assuming, since a notebook run headlessly by nbclient from the
# repo root would otherwise compute the wrong parent.
REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO))

RUNS_DIR = REPO / "data" / "tile_classifier"
SPLIT_MANIFEST = REPO / "data" / "splits" / "split_manifest.json"


class Checks:
    """Collects assertion results so one cell reports every check, not just the first.

    A bare `assert` stops at the first failure and hides how many other things are also
    wrong, which is the wrong trade for a verification pass: the useful output is the
    full list. Failures are still fatal -- `report()` raises at the end of the cell.
    """

    def __init__(self, title):
        self.title = title
        self.rows = []

    def expect(self, ok, label, detail=""):
        self.rows.append((bool(ok), label, detail))
        return bool(ok)

    def equal(self, observed, expected, label, tolerance=0.0):
        if isinstance(observed, float) and isinstance(expected, float):
            ok = (
                np.isclose(observed, expected, rtol=0, atol=tolerance)
                if tolerance
                else observed == expected
            )
        else:
            ok = observed == expected
        return self.expect(ok, label, f"observed={observed!r} expected={expected!r}")

    def report(self):
        failed = [r for r in self.rows if not r[0]]
        print(f"{self.title}: {len(self.rows) - len(failed)}/{len(self.rows)} passed")
        for ok, label, detail in self.rows:
            if not ok:
                print(f"  FAIL  {label}\n        {detail}")
        if failed:
            raise AssertionError(
                f"{self.title}: {len(failed)} of {len(self.rows)} checks failed"
            )
        print("  all passed")


def load_run(results_path):
    """One evaluated run: its results JSON plus the predictions it was scored from.

    A run directory holds one results file per evaluation, so a rescored run appears
    twice -- once under its training labelling and once under the common yardstick. Both
    are separate units of verification and both are loaded.
    """
    results = json.loads(results_path.read_text())
    suffix = results_path.name[len("results") : -len(".json")]
    predictions = results_path.parent / f"predictions{suffix}.csv"
    return {
        "key": f"{results_path.parent.name}{suffix}",
        "dir": results_path.parent,
        "results_path": results_path,
        "results": results,
        "predictions_path": predictions,
        # float_precision="round_trip" is load-bearing, not tidiness. pandas' default
        # CSV float parser is one ulp lossy: the threshold 0.31040725111961365 comes
        # back as 0.3104072511196136, which falls just BELOW the recorded operating
        # point, so the tile that defined the threshold stops being predicted positive.
        # The visible symptom is a one-tile disagreement in val precision/recall/F1/
        # accuracy/balanced accuracy/MCC in the fourth decimal, on exactly the runs
        # whose boundary tile is positive -- which reads like the results file is wrong
        # when it is the reader that is. AUPRC and ROC AUC are unaffected, since they
        # do not depend on the cut, and that asymmetry is the tell.
        "predictions": pd.read_csv(predictions, float_precision="round_trip"),
    }


def short_label(results):
    """A name that stays distinct after truncation.

    The run_id prefix does not: a rescored evaluation shares its parent's directory
    name and differs only in a trailing suffix, so truncating to a fixed width renders
    the two identically and a table silently appears to repeat a row.
    """
    name = {"resnet18": "rn18", "frozen_encoder": "phikon"}[results["architecture"]]
    label = f"{name}@{results['tile_size']} train{results['min_oocyte_area_fraction']:.2f}"
    if results["is_rescored"]:
        label += f" ->eval{results['eval_min_oocyte_area_fraction']:.2f}"
    return label


RUNS = [load_run(p) for p in sorted(RUNS_DIR.glob("*/results*.json"))]

assert RUNS, f"no results files under {RUNS_DIR}"
print(f"repo   {REPO}")
print(f"loaded {len(RUNS)} evaluated runs from {len(list(RUNS_DIR.iterdir()))} run directories\n")
for run in RUNS:
    r = run["results"]
    marker = " [RESCORED]" if r.get("is_rescored") else ""
    print(
        f"  {r['architecture']:<15} {r['tile_size']:>5}px  "
        f"train_area={r['min_oocyte_area_fraction']:.2f}  "
        f"eval_area={r.get('eval_min_oocyte_area_fraction')}  "
        f"{len(run['predictions']):>6} tiles{marker}"
    )

## Check 1 — the results schema matches §7 exactly (§12 item 1)

§12 item 1 asks for an *exact* key set, not a superset: an undocumented field is a failed check,
because a field nobody wrote down is a field nobody maintains. The expected keys below are
transcribed from §7 of the plan and are the only place in this notebook where the plan is treated
as authoritative — everything after this re-derives values instead.

This check has already earned its keep twice. §7 was found to have drifted behind the artifacts
during Step 8, and again during Step 10 when it was missing `eval_min_oocyte_area_fraction`,
`is_rescored` and `metrics.*.n_specimens`.

In [ ]:
"""Check 1: every results file carries exactly the keys §7 documents, at every level."""

SCHEMA_TOP_LEVEL = {
    "run_id", "architecture", "encoder_name", "encoder_licence", "tile_size", "seed",
    "label_rule", "min_oocyte_area_fraction", "eval_min_oocyte_area_fraction",
    "is_rescored", "n_ambiguous_excluded", "decision_threshold", "threshold_selection",
    "positive_fraction_per_batch", "positive_fraction_observed", "hard_negative_mining",
    "warmup_epochs", "hard_negative_pool_fraction", "hard_negative_share", "epochs_run",
    "best_epoch", "early_stopped", "early_stopped_epoch", "hyperparameters",
    "bootstrap_samples", "subset_caps", "is_subset_run", "split_manifest_version",
    "split_manifest_sha256", "tiling_provenance", "metrics", "checkpoint_path",
    "predictions_path", "batch_composition_log_path",
}
SCHEMA_TRAIN = {"n_tiles", "n_positive", "n_slides", "n_specimens", "max_train_slides"}
SCHEMA_SCORED = {
    "n_tiles", "n_positive", "n_slides", "n_specimens", "auprc", "roc_auc", "precision",
    "recall", "f1", "accuracy", "balanced_accuracy", "mcc", "trivial_baseline",
    "confidence_interval_95",
}
SCHEMA_HYPERPARAMETERS = {
    "batch_size", "lr", "weight_decay", "head_hidden_dim", "head_dropout",
}
SCHEMA_SUBSET_CAPS = {"max_train_slides", "max_batches_per_epoch", "max_val_tiles"}
SCHEMA_CI = {"auprc", "f1", "recall"}

checks = Checks("Check 1 -- schema")

for run in RUNS:
    r, key = run["results"], run["key"]
    checks.equal(set(r), SCHEMA_TOP_LEVEL, f"{key}: top-level keys")
    checks.equal(set(r["metrics"]), {"train", "val", "test"}, f"{key}: metric splits")
    checks.equal(set(r["metrics"]["train"]), SCHEMA_TRAIN, f"{key}: metrics.train keys")
    checks.equal(set(r["hyperparameters"]), SCHEMA_HYPERPARAMETERS, f"{key}: hyperparameters")
    checks.equal(set(r["subset_caps"]), SCHEMA_SUBSET_CAPS, f"{key}: subset_caps")
    checks.equal(set(r["n_ambiguous_excluded"]), {"train", "val", "test"}, f"{key}: ambiguous")
    for split in ("val", "test"):
        block = r["metrics"][split]
        checks.equal(set(block), SCHEMA_SCORED, f"{key}: metrics.{split} keys")
        checks.equal(
            set(block["confidence_interval_95"]), SCHEMA_CI, f"{key}: {split} CI keys"
        )
        checks.equal(
            set(block["trivial_baseline"]),
            {"all_negative_accuracy", "positive_rate"},
            f"{key}: {split} trivial_baseline keys",
        )
    # A Step 9 result must not be mistakable for a smoke run: §7 added these fields
    # precisely so a capped run announces itself.
    checks.equal(r["is_subset_run"], False, f"{key}: is_subset_run")
    checks.equal(
        set(run["predictions"].columns),
        {"tile_id", "stem", "cut_name", "split", "tile_size", "label",
         "oocyte_area_fraction", "prob"},
        f"{key}: prediction columns",
    )
    # The paths a results file advertises have to exist, or §12 items 2 and 3 cannot run.
    for field in ("checkpoint_path", "predictions_path", "batch_composition_log_path"):
        target = REPO / r[field]
        checks.expect(target.exists(), f"{key}: {field} exists", f"missing {target}")

checks.report()

## Check 2 — every reported metric re-derives from `predictions.csv` (§12 item 2)

The point of §12 item 2 is that §11's numbers must not be believed on the strength of the results
file that produced them. Here the eight threshold-free and threshold-bound metrics are recomputed
for both splits from the per-tile CSV, using the recorded `decision_threshold`, and compared to
nine decimal places.

Also checked, because a metric that re-derives from the wrong rows is still wrong: the tile and
positive counts, the trivial all-negative baseline the plan insists is quoted beside any accuracy,
and the slide and specimen counts the confidence intervals rest on.

In [ ]:
"""Check 2: recompute every val and test metric from the per-tile CSV, to 9 dp."""

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src.modeling.evaluate_tile_classifier import specimen_of

TOLERANCE = 1e-9


def recompute(frame, threshold):
    """The metric block a split should carry, derived only from the CSV rows."""
    targets = (frame["label"] == "positive").astype(int).to_numpy()
    probabilities = frame["prob"].to_numpy()
    predicted = (probabilities >= threshold).astype(int)
    return {
        "auprc": float(average_precision_score(targets, probabilities)),
        "roc_auc": float(roc_auc_score(targets, probabilities)),
        "precision": float(precision_score(targets, predicted, zero_division=0)),
        "recall": float(recall_score(targets, predicted, zero_division=0)),
        "f1": float(f1_score(targets, predicted, zero_division=0)),
        "accuracy": float(accuracy_score(targets, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(targets, predicted)),
        "mcc": float(matthews_corrcoef(targets, predicted)),
    }


checks = Checks("Check 2 -- metrics re-derived from predictions.csv")

for run in RUNS:
    r, key = run["results"], run["key"]
    threshold = r["decision_threshold"]
    for split in ("val", "test"):
        frame = run["predictions"].loc[run["predictions"]["split"] == split]
        reported = r["metrics"][split]
        checks.expect(len(frame) > 0, f"{key}/{split}: rows present")

        for name, value in recompute(frame, threshold).items():
            checks.equal(value, reported[name], f"{key}/{split}: {name}", TOLERANCE)

        checks.equal(len(frame), reported["n_tiles"], f"{key}/{split}: n_tiles")
        checks.equal(
            int((frame["label"] == "positive").sum()),
            reported["n_positive"],
            f"{key}/{split}: n_positive",
        )
        checks.equal(
            frame["stem"].nunique(), reported["n_slides"], f"{key}/{split}: n_slides"
        )
        checks.equal(
            frame["stem"].map(specimen_of).nunique(),
            reported["n_specimens"],
            f"{key}/{split}: n_specimens",
        )

        # The floor accuracy has to be read against. Recording it is the whole reason
        # §7 demotes accuracy: at these positive rates it flatters a useless model.
        positive_rate = float((frame["label"] == "positive").mean())
        checks.equal(
            positive_rate,
            reported["trivial_baseline"]["positive_rate"],
            f"{key}/{split}: trivial positive_rate",
            TOLERANCE,
        )
        checks.equal(
            1.0 - positive_rate,
            reported["trivial_baseline"]["all_negative_accuracy"],
            f"{key}/{split}: trivial all_negative_accuracy",
            TOLERANCE,
        )

        # Ambiguous tiles are excluded before scoring, so none may survive into the CSV.
        checks.equal(
            int((frame["label"] == "ambiguous").sum()), 0, f"{key}/{split}: no ambiguous rows"
        )

checks.report()

## Check 3 — the sampler delivered the ratio it was asked for (§12 item 3)

§5.3 forces a positive fraction per training batch, and §7 records both the requested value and
the realised one precisely so the two can be compared rather than assumed equal. `results.json`
reports the realised figure, but reporting it is not the same as its being true, so it is
recomputed here directly from `batch_composition.jsonl`.

Three separate things are asserted, because they can fail independently:

1. the realised fraction in `results.json` matches the log it claims to summarise;
2. the realised fraction matches the **requested** fraction, to within one tile of a batch —
   a last partial batch cannot hold an exact quarter of 64, so the tolerance is real rather than
   slack;
3. every record's own `requested_positive_fraction` matches the run's configured value, which
   catches a run whose flag changed partway through.

Hard-negative mining is also cross-checked: the log must show mined negatives **only** after the
warm-up epochs, and none at all in a run configured without mining.

In [ ]:
"""Check 3: realised batch composition against the requested ratio and the log."""

checks = Checks("Check 3 -- batch composition")
composition_rows = []

for run in RUNS:
    r, key = run["results"], run["key"]
    log_path = REPO / r["batch_composition_log_path"]
    records = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
    checks.expect(len(records) > 0, f"{key}: composition log is non-empty")

    positives = sum(rec["n_positive"] for rec in records)
    batched = sum(rec["batch_size"] for rec in records)
    observed = positives / batched
    requested = r["positive_fraction_per_batch"]

    checks.equal(observed, r["positive_fraction_observed"], f"{key}: observed matches log", 1e-12)

    # One tile of a 64-tile batch is the finest the sampler can resolve, so that is the
    # honest tolerance; anything wider would let a genuinely mis-specified sampler pass.
    batch_size = r["hyperparameters"]["batch_size"]
    checks.equal(observed, requested, f"{key}: observed matches requested", 1.0 / batch_size)

    checks.expect(
        all(rec["requested_positive_fraction"] == requested for rec in records),
        f"{key}: every batch requested the configured fraction",
        f"distinct values {sorted({rec['requested_positive_fraction'] for rec in records})}",
    )
    checks.expect(
        all(rec["n_positive"] + rec["n_negative"] == rec["batch_size"] for rec in records),
        f"{key}: batch counts sum to batch_size",
    )

    # Mining is a warm-up-gated behaviour, so the log has to show it switching on when
    # §5.3 says it does -- and never in a run that disabled it.
    mined_epochs = sorted({rec["epoch"] for rec in records if rec["n_hard_negative"] > 0})
    if r["hard_negative_mining"]:
        checks.expect(
            not mined_epochs or min(mined_epochs) > r["warmup_epochs"],
            f"{key}: mining starts after warm-up",
            f"warmup={r['warmup_epochs']} first mined epoch={mined_epochs[:1]}",
        )
    else:
        checks.equal(mined_epochs, [], f"{key}: no mining when disabled")

    composition_rows.append({
        "run": key,
        "requested": requested,
        "observed": round(observed, 6),
        "batches": len(records),
        "epochs": max(rec["epoch"] for rec in records),
        "mining": r["hard_negative_mining"],
        "mined_epochs": f"{min(mined_epochs)}-{max(mined_epochs)}" if mined_epochs else "-",
    })

checks.report()
print()
print(pd.DataFrame(composition_rows).to_string(index=False))

## Check 4 — every run was scored against the current split (§12 item 4)

§7 records `split_manifest_sha256` so that this is a hash comparison and not an act of faith. If
the manifest on disk has been regenerated since a run was evaluated, that run's val and test
figures describe a split that no longer exists, and nothing else in the artifact would say so.

The slide-to-split assignment the predictions actually use is also checked against the manifest,
which catches the narrower failure where the hash matches but the evaluation read the split from
somewhere else.

In [ ]:
"""Check 4: the recorded manifest hash is the hash of the manifest on disk."""

import hashlib

checks = Checks("Check 4 -- split manifest")

manifest_bytes = SPLIT_MANIFEST.read_bytes()
manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
manifest = json.loads(manifest_bytes)
assignment = {stem: entry["split"] for stem, entry in manifest["slides"].items()}

print(f"{SPLIT_MANIFEST.relative_to(REPO)}  sha256 {manifest_sha}")
print(f"  {len(assignment)} slides: " + ", ".join(
    f"{split}={sum(1 for v in assignment.values() if v == split)}"
    for split in ("train", "val", "test")
))

for run in RUNS:
    r, key = run["results"], run["key"]
    checks.equal(r["split_manifest_sha256"], manifest_sha, f"{key}: manifest sha256")
    checks.equal(r["split_manifest_version"], "v1", f"{key}: manifest version")

    # The hash proving the file is unchanged does not prove the evaluation read it, so
    # the per-tile split column is compared against the manifest assignment directly.
    observed = run["predictions"].groupby("stem")["split"].agg(
        lambda values: values.unique().tolist()
    )
    checks.expect(
        all(len(splits) == 1 for splits in observed),
        f"{key}: each slide sits in one split",
        f"{[s for s, v in observed.items() if len(v) > 1]}",
    )
    mismatched = {
        stem: (splits[0], assignment.get(stem))
        for stem, splits in observed.items()
        if assignment.get(stem) != splits[0]
    }
    checks.equal(mismatched, {}, f"{key}: prediction splits match the manifest")

checks.report()

## Check 5 — the labels reproduce from the geometry (§12 item 5)

This is the only check that goes back past the prediction log to the annotations themselves. The
tile index is rebuilt from `data/tiles/` and `data/cuts/` — polygon intersections and all — and the
coverage band is re-applied at each run's recorded threshold. If the recorded `n_positive` does
not fall out of that, either the threshold in the file is not the threshold that was used or the
labelling has moved underneath the results.

The subtlety §12 item 5 warns about is which threshold governs which block, and getting it
backwards makes correct files look wrong:

- `metrics.val` and `metrics.test`, and all three `n_ambiguous_excluded` counts, follow
  **`eval_min_oocyte_area_fraction`** — the labelling the metrics were computed against;
- `metrics.train` follows **`min_oocyte_area_fraction`** — the labelling the model trained under.

For the five runs that were not rescored the two coincide, which is exactly why a check that used
the wrong field would still pass on most of the corpus and fail only on the sweep.

Rebuilding the index parses every tile manifest and every annotation file, so this cell takes
about a minute per tile size. Each size is built once and relabelled, since coverage does not
depend on the threshold.

In [ ]:
"""Check 5: rebuild the labels from geometry and reproduce the recorded counts."""

import time

from src.modeling.tile_index import load_tile_index
from src.modeling.tile_labels import label_for_area_fraction

checks = Checks("Check 5 -- labels from geometry")

# One index per tile size, built once. The coverage fraction a tile carries is a
# property of the geometry, not of the threshold, so relabelling is a pure function of
# that column and the index does not need rebuilding per threshold.
INDEX_BY_SIZE = {}
for tile_size in sorted({run["results"]["tile_size"] for run in RUNS}):
    started = time.time()
    INDEX_BY_SIZE[tile_size] = load_tile_index(tile_sizes=(tile_size,))
    print(
        f"built {tile_size:>4}px index: {len(INDEX_BY_SIZE[tile_size]):>6} tiles "
        f"in {time.time() - started:.0f}s"
    )


def relabel(index, threshold):
    """The label column this index would carry at `threshold`."""
    return index["oocyte_area_fraction"].map(
        lambda fraction: label_for_area_fraction(fraction, threshold)
    )


for run in RUNS:
    r, key = run["results"], run["key"]
    index = INDEX_BY_SIZE[r["tile_size"]]

    scoring_threshold = r["eval_min_oocyte_area_fraction"]
    if scoring_threshold is None:
        scoring_threshold = r["min_oocyte_area_fraction"]
    checks.equal(
        bool(r["is_rescored"]),
        scoring_threshold != r["min_oocyte_area_fraction"],
        f"{key}: is_rescored agrees with the two thresholds",
    )
    checks.equal(r["label_rule"], "area", f"{key}: label rule")

    scored_labels = relabel(index, scoring_threshold)
    for split in ("train", "val", "test"):
        rows = index["split"] == split
        checks.equal(
            int((scored_labels[rows] == "ambiguous").sum()),
            r["n_ambiguous_excluded"][split],
            f"{key}: n_ambiguous_excluded[{split}] at coverage {scoring_threshold}",
        )

    # val and test follow the scoring threshold ...
    for split in ("val", "test"):
        rows = (index["split"] == split) & (scored_labels != "ambiguous")
        checks.equal(
            int(rows.sum()), r["metrics"][split]["n_tiles"], f"{key}: {split} n_tiles"
        )
        checks.equal(
            int((scored_labels[rows] == "positive").sum()),
            r["metrics"][split]["n_positive"],
            f"{key}: {split} n_positive at coverage {scoring_threshold}",
        )

    # ... while train follows the labelling the model was actually trained under.
    trained_labels = relabel(index, r["min_oocyte_area_fraction"])
    rows = (index["split"] == "train") & (trained_labels != "ambiguous")
    checks.equal(
        int(rows.sum()), r["metrics"]["train"]["n_tiles"], f"{key}: train n_tiles"
    )
    checks.equal(
        int((trained_labels[rows] == "positive").sum()),
        r["metrics"]["train"]["n_positive"],
        f"{key}: train n_positive at coverage {r['min_oocyte_area_fraction']}",
    )

    # The rebuilt labels must also agree tile by tile with the prediction log, not just
    # in aggregate: equal counts can hide two tiles swapping classes.
    expected = pd.Series(scored_labels.values, index=index["tile_id"].values)
    logged = run["predictions"].set_index("tile_id")["label"]
    checks.equal(
        int((expected.reindex(logged.index) != logged).sum()),
        0,
        f"{key}: per-tile labels match the prediction log",
    )

checks.report()

## Check 6 — the confidence intervals resample specimens (§12 item 6)

§12 item 6 asks for this to be established from the evaluation code rather than from the schema.
The strongest available form of that is reproduction: the specimen-clustered bootstrap is
reimplemented here from the §12 description alone — draw whole specimens with replacement, skip
single-class resamples, take the 2.5th and 97.5th percentiles — and if the intervals in
`results.json` come out of it bit for bit, the production code is doing what item 6 describes.

**An honest limitation, stated rather than hidden.** On the current 19/4/3 split val and test each
hold exactly one slide per specimen, so specimen grouping and slide grouping coincide and a
slide-level control cannot distinguish them here. That coincidence is itself asserted below, so
that it is a recorded fact rather than an assumption; it stops holding under M7b's grouped folds,
where the distinction starts to bite.

What *can* be distinguished now is the tile-level bootstrap the plan rejects as anticonservative.
It is computed as a negative control, and the width ratio printed at the end is the quantitative
form of the argument in §7: tiles overlap by 20% and one oocyte spans several, so resampling tiles
treats correlated rows as independent evidence and reports an interval the data does not support.

In [ ]:
"""Check 6: reproduce the published intervals by resampling specimens independently."""

checks = Checks("Check 6 -- specimen-clustered bootstrap")

# specimen_of is the grouping rule the intervals rest on, so it is exercised on the case
# the plan names before anything is built on top of it.
checks.equal(specimen_of("CHN_SP_5_22-24"), "CHN_SP_5", "specimen_of drops the cut range")
checks.equal(specimen_of("CHN_SP_5_25-27"), "CHN_SP_5", "serial sections share a specimen")


def _interval_from(collected):
    return {
        name: [float(v) for v in np.percentile(values, [2.5, 97.5])]
        for name, values in collected.items()
    }


def clustered_intervals(frame, threshold, groups, draws, seed):
    """95% intervals for AUPRC, F1 and recall, resampling whole groups.

    Reimplemented from the §12 description rather than imported, so that agreement with
    the published numbers is evidence about the production code instead of a tautology.
    """
    targets = (frame["label"] == "positive").astype(int).to_numpy()
    probabilities = frame["prob"].to_numpy()
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    rows_by_group = {g: np.flatnonzero(groups == g) for g in unique_groups}

    collected = {"auprc": [], "f1": [], "recall": []}
    for _ in range(draws):
        drawn = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        rows = np.concatenate([rows_by_group[g] for g in drawn])
        sample_targets = targets[rows]
        if len(np.unique(sample_targets)) < 2:
            continue
        sample_probabilities = probabilities[rows]
        predicted = (sample_probabilities >= threshold).astype(int)
        collected["auprc"].append(
            float(average_precision_score(sample_targets, sample_probabilities))
        )
        collected["f1"].append(float(f1_score(sample_targets, predicted, zero_division=0)))
        collected["recall"].append(
            float(recall_score(sample_targets, predicted, zero_division=0))
        )
    return _interval_from(collected)


def tile_intervals(frame, threshold, draws, seed):
    """The same intervals with every tile its own group -- the rejected alternative.

    Kept separate rather than expressed as `clustered_intervals(groups=arange(n))`,
    which is the same arithmetic but builds an n-entry group map by scanning the whole
    array once per group and then concatenates n single-element arrays per draw. At
    7,597 val tiles that is quadratic and does not finish.
    """
    targets = (frame["label"] == "positive").astype(int).to_numpy()
    probabilities = frame["prob"].to_numpy()
    n = len(frame)
    rng = np.random.default_rng(seed)

    collected = {"auprc": [], "f1": [], "recall": []}
    for _ in range(draws):
        rows = rng.integers(0, n, size=n)
        sample_targets = targets[rows]
        if len(np.unique(sample_targets)) < 2:
            continue
        sample_probabilities = probabilities[rows]
        predicted = (sample_probabilities >= threshold).astype(int)
        collected["auprc"].append(
            float(average_precision_score(sample_targets, sample_probabilities))
        )
        collected["f1"].append(float(f1_score(sample_targets, predicted, zero_division=0)))
        collected["recall"].append(
            float(recall_score(sample_targets, predicted, zero_division=0))
        )
    return _interval_from(collected)


width_rows = []
for run in RUNS:
    r, key = run["results"], run["key"]
    for split in ("val", "test"):
        frame = run["predictions"].loc[run["predictions"]["split"] == split].reset_index(drop=True)
        specimens = frame["stem"].map(specimen_of).to_numpy()

        # The coincidence that makes a slide-level control uninformative on this split.
        # Asserted so that if a future split breaks it, this notebook says so.
        checks.equal(
            len(np.unique(specimens)),
            frame["stem"].nunique(),
            f"{key}/{split}: one slide per specimen on this split",
        )

        reproduced = clustered_intervals(
            frame, r["decision_threshold"], specimens, r["bootstrap_samples"], r["seed"]
        )
        for name, interval in reproduced.items():
            published = r["metrics"][split]["confidence_interval_95"][name]
            checks.equal(interval[0], published[0], f"{key}/{split}: {name} CI lower", 1e-12)
            checks.equal(interval[1], published[1], f"{key}/{split}: {name} CI upper", 1e-12)

        # Negative control: one group per tile, which is the assumption of independence
        # the plan rejects. Recorded, not asserted against a threshold -- the claim is
        # that it is narrower, and the numbers are the evidence.
        per_tile = tile_intervals(
            frame,
            r["decision_threshold"],
            200,  # fewer draws: this control is illustrative, and 200 is enough to show the gap
            r["seed"],
        )
        for name in ("auprc", "f1", "recall"):
            clustered_width = reproduced[name][1] - reproduced[name][0]
            tile_width = per_tile[name][1] - per_tile[name][0]
            width_rows.append({
                "run": short_label(r),
                "split": split,
                "metric": name,
                "specimen_width": round(clustered_width, 4),
                "tile_width": round(tile_width, 4),
                # 0/0 is undefined, not infinite. ResNet-18 @1024 catches all 135 test
                # positives at its operating point, so recall is 1.0 in every resample
                # under either grouping and both intervals collapse to [1.0, 1.0]. That
                # is a real property of the run, but it carries no information about
                # clustering, so it is excluded from the mean rather than dominating it.
                "ratio": (
                    round(clustered_width / tile_width, 1) if tile_width else float("nan")
                ),
            })

checks.report()
print()
widths = pd.DataFrame(width_rows)
print("Interval width, specimen-clustered vs tile-level (the rejected alternative):")
print(widths.groupby(["split", "metric"])[["specimen_width", "tile_width", "ratio"]].mean().round(3).to_string())
undefined = int(widths["ratio"].isna().sum())
if undefined:
    print(f"\n({undefined} of {len(widths)} ratios undefined, both intervals zero-width)")

## Check 7 — the operating point is the exact validation maximiser, applied unchanged to test

Not one of §12's seven items, but the finding that produced them. §8's Step 8 note records that a
standing 0.5 cut is indefensible here — the sampler forces 25% positives per training batch while
val and test run at their natural rates, so the probabilities are on a different scale from the
data they are scored on by construction. Two things therefore have to hold, and neither is visible
in the metrics themselves:

1. `decision_threshold` really is the F1 maximiser over **every** distinct validation probability,
   not the best point on a grid or a subsample. Step 9 found the earlier 512-point quantile
   subsample skipped most of the real cut points, so the sweep is re-run here at full resolution —
   in the opposite sort order from the production implementation, so agreement is not an artifact
   of shared code — and cross-checked against `sklearn.f1_score` at sampled candidates.
2. Test is scored at that same value and never picks its own. The test-optimal threshold is
   computed alongside and reported; the gap between test F1 at the val-selected point and test F1
   at its own optimum is the size of the advantage that selecting on test would have bought, which
   is the number that makes the discipline concrete.

Ties are required to resolve to the **lower** threshold, which favours recall: for a screening
stage feeding detection, missing an oocyte is the costlier error.

In [ ]:
"""Check 7: the recorded threshold maximises validation F1 exactly, and test inherits it."""

checks = Checks("Check 7 -- operating point")


def f1_sweep(targets, probabilities):
    """F1 at every distinct probability, as (candidates ascending, scores).

    Deliberately built from an ASCENDING sort and suffix sums, where the production
    implementation uses a descending sort and prefix sums. The two reach the same
    numbers by different arithmetic, so agreement is evidence rather than a shared bug.
    """
    order = np.argsort(probabilities, kind="stable")
    sorted_probabilities = probabilities[order]
    sorted_targets = targets[order].astype(np.int64)
    total_positive = int(sorted_targets.sum())

    # A cut at sorted_probabilities[i] predicts positive for i..n-1, so suffix sums give
    # TP and the predicted-positive count directly.
    suffix_positive = np.concatenate(
        [np.cumsum(sorted_targets[::-1])[::-1], [0]]
    )
    suffix_count = np.arange(len(sorted_probabilities), -1, -1)

    starts = np.concatenate(
        [[0], np.flatnonzero(np.diff(sorted_probabilities)) + 1]
    ).astype(np.int64)
    candidates = sorted_probabilities[starts]
    true_positive = suffix_positive[starts]
    predicted_positive = suffix_count[starts]

    denominator = predicted_positive + total_positive
    scores = np.divide(
        2 * true_positive,
        denominator,
        out=np.zeros(len(starts), dtype=float),
        where=denominator > 0,
    )
    return candidates, scores


threshold_rows = []
rng = np.random.default_rng(0)

for run in RUNS:
    r, key = run["results"], run["key"]
    checks.equal(r["threshold_selection"], "max_val_f1", f"{key}: threshold_selection")

    frames = {
        split: run["predictions"].loc[run["predictions"]["split"] == split]
        for split in ("val", "test")
    }
    targets = {
        split: (frame["label"] == "positive").astype(int).to_numpy()
        for split, frame in frames.items()
    }
    probabilities = {split: frame["prob"].to_numpy() for split, frame in frames.items()}

    candidates, scores = f1_sweep(targets["val"], probabilities["val"])
    best = float(scores.max())
    # Ties resolve to the lower threshold; candidates ascend, so that is the FIRST maximum.
    chosen = float(candidates[int(np.argmax(scores))])

    checks.equal(chosen, r["decision_threshold"], f"{key}: threshold is the exact maximiser", 0.0)
    checks.equal(
        best,
        r["metrics"]["val"]["f1"],
        f"{key}: val F1 at that threshold is the maximum",
        1e-12,
    )

    # Independent spot-check of the vectorised sweep against sklearn, so a bug shared by
    # both cumulative implementations would still be caught.
    sampled = rng.choice(len(candidates), size=min(200, len(candidates)), replace=False)
    disagreements = [
        (float(candidates[i]), float(scores[i]), reference)
        for i in sampled
        if abs(
            scores[i]
            - (
                reference := f1_score(
                    targets["val"],
                    (probabilities["val"] >= candidates[i]).astype(int),
                    zero_division=0,
                )
            )
        )
        > 1e-12
    ]
    checks.equal(disagreements, [], f"{key}: sweep agrees with sklearn at 200 candidates")

    # No candidate may beat the chosen one, and 0.5 must not be silently assumed.
    checks.expect(
        bool((scores <= best + 1e-12).all()), f"{key}: no candidate beats the chosen point"
    )

    test_candidates, test_scores = f1_sweep(targets["test"], probabilities["test"])
    test_own_best = float(test_scores.max())
    threshold_rows.append({
        "run": short_label(r),
        "val_thr": round(r["decision_threshold"], 4),
        "val_f1": round(r["metrics"]["val"]["f1"], 4),
        "f1@0.5": round(
            f1_score(
                targets["val"], (probabilities["val"] >= 0.5).astype(int), zero_division=0
            ),
            4,
        ),
        "test_f1": round(r["metrics"]["test"]["f1"], 4),
        "test_own_thr": round(float(test_candidates[int(np.argmax(test_scores))]), 4),
        "test_own_f1": round(test_own_best, 4),
        "forgone": round(test_own_best - r["metrics"]["test"]["f1"], 4),
    })

    # Selecting on test can only ever help test, so a negative gap would mean the test
    # figure was not produced at the validation-selected point.
    checks.expect(
        test_own_best >= r["metrics"]["test"]["f1"] - 1e-12,
        f"{key}: test scored at the val-selected point, not its own optimum",
        f"test_f1={r['metrics']['test']['f1']} test_optimum={test_own_best}",
    )

checks.report()
print()
print("Operating points (`forgone` = the F1 that selecting on test would have bought):")
print(pd.DataFrame(threshold_rows).to_string(index=False))

## Summary — the tables §11 quotes

Everything above asserts. This cell only reports, and every figure in it has already been
re-derived from `predictions.csv` by Check 2, so these are the numbers §11 should carry.

Three tables:

1. **Main grid** — the four runs that vary architecture and tile scale at the common 0.05 coverage
   threshold. These are the runs the tile-scale recommendation rests on, and their intervals
   overlap.
2. **Coverage sweep, common yardstick** — the three 512 px runs trained at 0.05, 0.10 and 0.25 and
   all scored against 0.05 labels. Scored against their own training thresholds these runs are not
   comparable, because a higher threshold removes the hardest positives from the exam as well as
   from the training set; rescoring is what makes the comparison about the model rather than about
   the difficulty of its own test.
3. **Leakage** — test performance split by whether the specimen also appears in train. §9 Step 9
   requires the recommendation to be marked provisional pending M7b's specimen-grouped folds, and
   this table is the evidence about how much the leakage is actually worth here.

In [ ]:
"""Summary tables. Reporting only -- every figure was asserted above."""

ENCODER_LABEL = {"resnet18": "ResNet-18", "frozen_encoder": "Phikon-v2"}


def describe(run):
    r = run["results"]
    return f"{ENCODER_LABEL[r['architecture']]} @{r['tile_size']}"


def interval(block, name):
    low, high = block["confidence_interval_95"][name]
    return f"[{low:.3f}, {high:.3f}]"


by_key = {run["key"]: run for run in RUNS}

main_grid = [
    run for run in RUNS
    if not run["results"]["is_rescored"] and run["results"]["min_oocyte_area_fraction"] == 0.05
]
rows = []
for run in sorted(main_grid, key=lambda run: (run["results"]["architecture"], run["results"]["tile_size"])):
    r = run["results"]
    test, val = r["metrics"]["test"], r["metrics"]["val"]
    rows.append({
        "model": describe(run),
        "epochs": r["epochs_run"],
        "best": r["best_epoch"],
        "thr": round(r["decision_threshold"], 4),
        "val AP": round(val["auprc"], 4),
        "test AP": round(test["auprc"], 4),
        "test AP 95% CI": interval(test, "auprc"),
        "test F1": round(test["f1"], 4),
        "test F1 95% CI": interval(test, "f1"),
        "test MCC": round(test["mcc"], 4),
        "floor acc": round(test["trivial_baseline"]["all_negative_accuracy"], 4),
        "acc": round(test["accuracy"], 4),
    })
print("1. Main grid -- architecture x tile scale, coverage 0.05")
print(pd.DataFrame(rows).to_string(index=False))

sweep = [
    run for run in RUNS
    if run["results"]["tile_size"] == 512
    and (run["results"]["eval_min_oocyte_area_fraction"] or 0.05) == 0.05
    and run["results"]["architecture"] == "frozen_encoder"
]
rows = []
for run in sorted(sweep, key=lambda run: run["results"]["min_oocyte_area_fraction"]):
    r = run["results"]
    test = r["metrics"]["test"]
    rows.append({
        "trained at": r["min_oocyte_area_fraction"],
        "scored at": r["eval_min_oocyte_area_fraction"],
        "rescored": r["is_rescored"],
        "test tiles": test["n_tiles"],
        "test pos": test["n_positive"],
        "test AP": round(test["auprc"], 4),
        "test AP 95% CI": interval(test, "auprc"),
        "test F1": round(test["f1"], 4),
        "test recall": round(test["recall"], 4),
    })
print("\n2. Coverage sweep, all scored against the common 0.05 yardstick (Phikon-v2 @512)")
print(pd.DataFrame(rows).to_string(index=False))

train_specimens = {
    specimen_of(stem) for stem, split in assignment.items() if split == "train"
}
rows = []
for run in sorted(main_grid, key=lambda run: (run["results"]["architecture"], run["results"]["tile_size"])):
    r = run["results"]
    frame = run["predictions"].loc[run["predictions"]["split"] == "test"].copy()
    frame["clean"] = ~frame["stem"].map(specimen_of).isin(train_specimens)
    for clean, group in frame.groupby("clean"):
        targets = (group["label"] == "positive").astype(int).to_numpy()
        if len(np.unique(targets)) < 2:
            continue
        probabilities = group["prob"].to_numpy()
        predicted = (probabilities >= r["decision_threshold"]).astype(int)
        rows.append({
            "model": describe(run),
            "specimens": "unseen in train" if clean else "also in train",
            "n_specimens": group["stem"].map(specimen_of).nunique(),
            "tiles": len(group),
            "pos": int(targets.sum()),
            "AP": round(float(average_precision_score(targets, probabilities)), 4),
            "F1": round(float(f1_score(targets, predicted, zero_division=0)), 4),
            "recall": round(float(recall_score(targets, predicted, zero_division=0)), 4),
        })
leakage = pd.DataFrame(rows)
print("\n3. Test performance by whether the specimen also appears in train")
print(leakage.to_string(index=False))
print(
    "\n   Train specimens: "
    + ", ".join(sorted(train_specimens))
    + f"\n   Test specimens:  "
    + ", ".join(sorted({specimen_of(s) for s, v in assignment.items() if v == "test"}))
)